# 11 · Migrating from Pydantic v1, Best Practices & Alternatives

**Module 15 – Pydantic: Data Validation & Settings** · Estimated time: 35 min

> Many tutorials, Stack Overflow answers and older codebases use **Pydantic v1** (before mid-2023). This lesson maps every common v1 API to v2, shows the deprecation warnings you'll see, and closes the module with best practices.

### 🎯 What you will learn
- ⚠️ The complete v1 → v2 rename table
- Running v1 code: `pydantic.v1`, deprecation warnings, the `bump-pydantic` tool
- Best practices for real projects
- When to use Pydantic vs dataclasses, `attrs`, `msgspec`, `TypedDict`

## 1. ⚠️ v1 → v2 cheat-sheet

| Pydantic v1 (deprecated) | ✅ Pydantic v2 |
|---|---|
| `model.dict()` | `model.model_dump()` |
| `model.json()` | `model.model_dump_json()` |
| `Model.parse_obj(data)` | `Model.model_validate(data)` |
| `Model.parse_raw(text)` | `Model.model_validate_json(text)` |
| `Model.parse_file(path)` | `Model.model_validate_json(Path(path).read_text())` |
| `Model.from_orm(obj)` | `Model.model_validate(obj)` + `from_attributes=True` |
| `Model.construct(...)` | `Model.model_construct(...)` |
| `model.copy(update=...)` | `model.model_copy(update=...)` |
| `Model.schema()` / `schema_json()` | `Model.model_json_schema()` |
| `Model.__fields__` | `Model.model_fields` |
| `model.__fields_set__` | `model.model_fields_set` |
| `Model.update_forward_refs()` | `Model.model_rebuild()` |
| `@validator("x")` | `@field_validator("x")` (+ `@classmethod`) |
| `@validator(..., pre=True)` | `@field_validator(..., mode="before")` |
| `@validator(..., each_item=True)` | `list[Annotated[T, AfterValidator(...)]]` |
| `@root_validator` / `(pre=True)` | `@model_validator(mode="after")` / `mode="before"` |
| `class Config:` | `model_config = ConfigDict(...)` |
| `orm_mode = True` | `from_attributes=True` |
| `allow_population_by_field_name` | `validate_by_name=True` (formerly `populate_by_name`) |
| `anystr_strip_whitespace` | `str_strip_whitespace` |
| `min_anystr_length` / `max_anystr_length` | `str_min_length` / `str_max_length` |
| `allow_mutation = False` | `frozen=True` |
| `underscore_attrs_are_private` | always private in v2 (`PrivateAttr`) |
| `json_encoders = {...}` | `@field_serializer` / `PlainSerializer` |
| `Field(regex=...)` | `Field(pattern=...)` |
| `Field(min_items=, max_items=)` | `Field(min_length=, max_length=)` |
| `Field(const=True)` | `Literal[...]` |
| `conint(gt=0)`, `constr(...)` | `Annotated[int, Field(gt=0)]`, `StringConstraints` |
| `from pydantic import BaseSettings` | `from pydantic_settings import BaseSettings` |
| `from pydantic.generics import GenericModel` | plain `BaseModel` with type parameters |
| `__root__: list[int]` | `RootModel[list[int]]` |
| `Optional[X]` meant "not required" | `X \| None` is **required** unless it has a default |

> ❗ That last row is the most common migration bug: in v1, `name: Optional[str]` had an implicit default of `None`; in v2 you must write `name: str | None = None`.

## 2. Deprecation warnings in action

In [1]:
import warnings
from pydantic import BaseModel, ConfigDict

class User(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    id: int
    name: str

u = User(id=1, name="Asha")
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    u.dict()
    u.json()
    User.parse_obj({"id": 2, "name": "Ravi"})
for w in caught:
    print(f"{w.category.__name__}: {str(w.message)[:90]}")

PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to 
PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.
PydanticDeprecatedSince20: The `parse_obj` method is deprecated; use `model_validate` instead. Deprecated in Pydantic


The old methods still work in v2 (with warnings) to ease migration, and will be removed in a future major version.

## 3. Migration strategy
1. Upgrade with tests in place; run with `-W error::DeprecationWarning` to find old APIs
2. Run **`bump-pydantic`** (`uvx bump-pydantic src/`), an official codemod that rewrites most v1 code automatically
3. Fix what remains by hand using the table above
4. If a dependency still needs v1, the v1 API is available as **`pydantic.v1`** (`from pydantic.v1 import BaseModel`) during the transition

In [2]:
import pydantic.v1 as pv1
print("v1 compatibility layer available:", hasattr(pv1, "BaseModel"))

v1 compatibility layer available: True


## 4. Best practices

**Design**
- ✅ Validate **at the boundaries** (HTTP, files, env, queues, LLM output); trust validated objects inside
- ✅ Separate **input** (`Create`/`Update`) and **output** (`Public`) models; never return ORM objects or secrets directly
- ✅ `extra="forbid"` on input models; `frozen=True` for value objects
- ✅ Reusable `Annotated` types for domain concepts (`Email`, `Money`, `Slug`, `Pincode`)
- ✅ Discriminated unions for polymorphic payloads
- ✅ `SecretStr` for secrets; `pydantic-settings` for configuration

**Validation logic**
- Keep validators small and pure; raise `ValueError` (or `PydanticCustomError` with stable error types)
- Cross-field rules in `@model_validator(mode="after")`
- Don't do I/O (DB/HTTP calls) inside validators; validate shape here, check business rules in services

**Performance**
- `model_validate_json` over `json.loads` + `model_validate`
- Create `TypeAdapter`s once; prefer discriminated unions
- Measure before reaching for `model_construct`

**Typing & tooling**
- Enable the **mypy Pydantic plugin** (`plugins = ["pydantic.mypy"]`) or use pyright, which understands Pydantic via `dataclass_transform` (Module 10 · 07)
- Generate JSON Schema/OpenAPI from models instead of writing them by hand

## 5. Alternatives: which tool when?

| Tool | Validation | Speed | Best for |
|---|---|---|---|
| `dataclasses` | ❌ | ⚡ | internal value objects |
| `TypedDict` | ❌ (static only) | ⚡ | typing existing dicts / JSON |
| **Pydantic** | ✅ rich, coercion, errors, schema | fast (Rust) | APIs, configs, external data, LLM I/O |
| `attrs` (+ `cattrs`) | optional validators | ⚡ | rich classes, (de)structuring |
| `msgspec` | ✅ strict | ⚡⚡ | very high-throughput serialization |
| `marshmallow` | ✅ (schemas separate from classes) | slower | older Flask-era codebases |

## 📝 Summary
- v2 renamed almost everything with a `model_` prefix, replaced `@validator`/`@root_validator` and `class Config`, and moved `BaseSettings` to `pydantic-settings`.
- Use `bump-pydantic`, deprecation warnings and `pydantic.v1` to migrate gradually.
- Validate at the edges, separate input/output models, reuse `Annotated` types, and keep validators pure.

## 🏋️ Practice: migrate this v1 code to v2

```python
from typing import Optional, List
from pydantic import BaseModel, validator, root_validator, BaseSettings

class Item(BaseModel):
    name: str
    tags: List[str] = []
    price: Optional[float]

    class Config:
        orm_mode = True
        anystr_strip_whitespace = True

    @validator("name")
    def name_not_empty(cls, v):
        assert v, "empty"
        return v

    @root_validator
    def check(cls, values):
        if values.get("price") is not None and values["price"] < 0:
            raise ValueError("negative price")
        return values

class Settings(BaseSettings):
    debug: bool = False

item = Item.parse_obj({"name": " pen ", "price": 3})
print(item.dict(), Item.schema())
```

<details><summary>Show solution</summary>

```python
from typing import Self
from pydantic import BaseModel, ConfigDict, field_validator, model_validator
from pydantic_settings import BaseSettings

class Item(BaseModel):
    model_config = ConfigDict(from_attributes=True, str_strip_whitespace=True)
    name: str
    tags: list[str] = []
    price: float | None = None          # v1's Optional implied a default

    @field_validator("name")
    @classmethod
    def name_not_empty(cls, v: str) -> str:
        if not v:
            raise ValueError("empty")
        return v

    @model_validator(mode="after")
    def check(self) -> Self:
        if self.price is not None and self.price < 0:
            raise ValueError("negative price")
        return self

class Settings(BaseSettings):
    debug: bool = False

item = Item.model_validate({"name": " pen ", "price": 3})
print(item.model_dump(), Item.model_json_schema()["required"])
```
</details>